In [6]:

import os
os.getcwd()
os.chdir("/gradution project")
os.getcwd()

'e:\\gradution project'

In [7]:
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

from src.similarity_model import preprocess_dataset
from src.similarity_model import train_embedding_engine
from src.similarity_model import search_by_text
from src.similarity_model import find_similar_projects
from Data.database.sql_connector import (
    load_preprocessed_projects,
    engine
)

print("All modules imported successfully")

All modules imported successfully


In [9]:
from sqlalchemy import create_engine
import urllib

SERVER = "innotrack-sql-server.database.windows.net"
DATABASE = "InnoTrackDB"
USERNAME = "innotrackadmin"
PASSWORD = "Innotrack@admin233"

params = urllib.parse.quote_plus(
    f"DRIVER={{ODBC Driver 18 for SQL Server}};"
    f"SERVER={SERVER};"
    f"DATABASE={DATABASE};"
    f"UID={USERNAME};"
    f"PWD={PASSWORD};"
    "Encrypt=yes;"
    "TrustServerCertificate=no;"
    "Connection Timeout=30;"
)

engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params}"
)

print("Engine created")

Engine created


In [ ]:
with engine.connect() as conn:

    tables = pd.read_sql(
        """
        SELECT TABLE_NAME
        FROM INFORMATION_SCHEMA.TABLES
        """,
        conn
    )

tables

,TABLE_NAME
0,Teams
1,ChatRooms
2,JoinRequests
3,Projects
4,TeamMembers
5,ChatMessages
6,Feedbacks
7,OriginalityReports
8,ProjectAttachments
9,ProjectTechnologies


In [ ]:
query = """
SELECT *
FROM Projects
"""

df_raw= pd.read_sql(query, engine)


print("Rows:", len(df_raw))
df_raw.tail()

FileNotFoundError: [Errno 2] No such file or directory: 'E:\\gradution project\\Data\\raw\\Graduation Projects - Sheet1.xlsx'

In [ ]:
clean_df = preprocess_dataset(df_raw)

print("Rows after cleaning:", len(clean_df))
clean_df.tail()

In [ ]:
print(type(clean_df.loc[0, "features"]))
print(clean_df.loc[0, "features"])

In [ ]:
print(clean_df.columns.tolist())

In [ ]:
object_cols = clean_df.select_dtypes(include="object").columns

for col in object_cols:
    clean_df[col] = clean_df[col].astype(str)

In [ ]:
clean_df.to_parquet("Data/processed/projects_clean.parquet", index=False)
clean_df.to_csv("Data/processed/projects_clean.csv", index=False)

print("Saved cleaned dataset")

In [ ]:
engine = train_embedding_engine(
    data_path="Data/processed/projects_clean.parquet"
)

print("Embedding trained successfully")

In [ ]:
from importlib import reload
import src.similarity_model as fs

reload(fs)

In [ ]:
from importlib import reload
import src.similarity_model as sim

reload(sim)

In [ ]:
final_results = find_similar_projects(
    title="Smart Library app",
    description="""
    Library chatbot book recommendation search engine 
    """,
    top_k=5
)
final_results

In [ ]:
! python -m src.similarity_model.evaluation